# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

A Croissant schema groups data into record sets (e.g., tables/objects of related records) with each column or field traceable by its `@id`.

In [ ]:
from pprint import pprint

# Display all available record sets and their fields (all by @id)
def overview_record_sets(ds):
    print("Available Record Sets:")
    recsets = ds.metadata.record_sets
    for rec in recsets:
        print(f"- RecordSet @id: {rec['@id']} (name={rec.get('name','')})")
        fields = rec.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        for fld in fields:
            print(f"   - Field @id: {fld['@id']} (name={fld.get('name','')}, dataType={fld.get('dataType','')})")
    return recsets

record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found; attempting to list data files.')
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"- Data File Distribution @id: {dist['@id']}, contentUrl: {dist.get('contentUrl', '')}")
    else:
        print('No data files found.')
else:
    _ = overview_record_sets(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

If no explicit record sets are declared, we attempt to extract from detected distributions (data files) directly.

In [ ]:
import itertools

# Helper: get all record set @id values, or else use distributions for direct loading
record_sets = dataset.metadata.record_sets

if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Fallback: load each data file (distribution)
    dist_ids = [dist['@id'] for dist in getattr(dataset.metadata, 'distribution', [])]
    record_set_ids = dist_ids
    print(f"No recordSet objects in schema, using distributions as record sets: {record_set_ids}")

dataframes = dict()

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for record set @id: {rs_id}")
            print(f"Fields (columns): {df.columns.tolist()[:10]}{'...' if len(df.columns)>10 else ''}")
        else:
            print(f"No records found for record set @id: {rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Show a sample record set
if dataframes:
    sample_rs_id = list(dataframes.keys())[0]
    print(f"\nSample column names for record set @id={sample_rs_id}:\n{dataframes[sample_rs_id].columns.tolist()}")
    dataframes[sample_rs_id].head()
else:
    print('No DataFrames extracted. Ensure data files are referenced in the schema.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

You must use column `@id` for reference. Example: to filter by 'Molecular Weight', find its field `@id` in the schema/column list printed above.

In [ ]:
# Choose a record set and numeric field using their @id
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to find a potentially useful numeric field; fallback to first
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
    if not numeric_candidates:
        from pandas.api.types import is_numeric_dtype
        # try to coerce any column to numeric if possible
        for col in df.columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().any():
                numeric_candidates.append(col)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # Use @id (column name)
        threshold = df[numeric_field_id].quantile(0.80)  # use a high threshold for demonstration

        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:\n", filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
        ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to select a non-numeric group field
        group_field = None
        cat_cols = df.select_dtypes(include=['object']).columns.tolist()
        group_candidates = [c for c in cat_cols if df[c].nunique() > 1 and df[c].nunique() < len(df)//2]
        if group_candidates:
            group_field = group_candidates[0]

        if group_field:
            grouped = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (using mean):")
            print(grouped.head())
    else:
        print('No numeric field detected for EDA.')
else:
    print('No DataFrame available for EDA section.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example using matplotlib to visualize a histogram and scatter plot for the selected numeric field and, if available, a second numeric field.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    # Try to find numeric fields
    numeric = df.select_dtypes(include='number').columns.tolist()
    if len(numeric) >= 1:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric[0]].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {numeric[0]}")
        plt.xlabel(numeric[0])
        plt.show()

    if len(numeric) >= 2:
        plt.figure(figsize=(6,6))
        plt.scatter(df[numeric[0]], df[numeric[1]], alpha=0.5)
        plt.xlabel(numeric[0])
        plt.ylabel(numeric[1])
        plt.title(f"{numeric[1]} vs. {numeric[0]}")
        plt.show()
else:
    print('No data to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was successfully loaded using the Croissant `mlcroissant` interface.
- All data extraction and manipulation was referenced by schema `@id` fields for accuracy and reproducibility.
- This workflow sets the foundation for further FAIR-compliant data science and enables transparent, automated reproducibility.
